# 第 3 章 物価指数と景気・雇用の指標: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-basics-programming/03-price-indexes-and-indicators.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
periods = ["2020", "2021", "2022"]
bread_prices = [100, 110, 121]      # パンの価格
milk_prices = [200, 180, 170]       # 牛乳の価格
bread_quantities = [10, 12, 14]     # パンの数量
milk_quantities = [5, 4, 3]         # 牛乳の数量
base_period = "2020"                # 物価指数の基準時
nominal_gdp = 550                   # 名目 GDP
deflator = 110                      # GDP デフレータ（基準時 = 100）
employed = 6700                     # 就業者
unemployed = 200                    # 完全失業者
base_month = "2025-01"              # 景気動向指数 CI の基準の月（= 100）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 物価指数・実質 GDP・失業率

In [ ]:
import pandas as pd

from lib.chapter03.indicators import unemployment_rate
from lib.chapter03.price_index import price_indexes, real_gdp

prices = pd.DataFrame({"bread": bread_prices, "milk": milk_prices}, index=periods)
quantities = pd.DataFrame(
    {"bread": bread_quantities, "milk": milk_quantities}, index=periods
)
table = price_indexes(prices, quantities, base=base_period)
print(table)
print("実質 GDP:", real_gdp(nominal_gdp, deflator))
print(f"完全失業率: {unemployment_rate(employed, unemployed):.2f}%")

## 図: 物価指数

In [ ]:
from lib.chapter03.plot import plot_price_indexes

plot_price_indexes(table)

## 景気動向指数（DI・CI）

In [ ]:
from pathlib import Path

from lib.chapter03.indicators import composite_index, diffusion_index, load_indicators

# アプリのディレクトリからでも、notebooks/ からでも、data/ を見つける
data_dir = Path("data") if Path("data").exists() else Path("../data")
indicators = load_indicators(data_dir / "ch03_indicators.csv")
di = diffusion_index(indicators)
ci = composite_index(indicators, base=base_month)
print(f"最後の月の CI（{base_month} = 100）: {ci.iloc[-1]:.2f}")

In [ ]:
from lib.chapter03.plot import plot_business_indexes

plot_business_indexes(di, ci)

## 試してみよう

- 数量を一定にする（`bread_quantities = [10, 10, 10]`、
  `milk_quantities = [5, 5, 5]`）と、
  ラスパイレス指数とパーシェ指数は、2022 年にどちらも 1.03 で一致します
  （もとは 1.03 と 1.102）。**2 つの指数が食い違うのは、数量が変わるから**です
- `nominal_gdp = 550`、`deflator = 110` のとき、実質 GDP は 500.0 です
  （デフレータを 100 にすると、名目と実質が同じ 550 になります）
- `base_month = "2025-06"` にすると、最後の月の CI は 97.49 になります
  （`2025-01` のときは 104.82）。基準の月を変えると、100 の位置が動きます